In [ ]:
import sys,os
notebook_dir = os.getcwd()  # Gets current working directory
parent_dir = os.path.abspath(os.path.join(notebook_dir, '..'))
sys.path.append(parent_dir)
sys.path.append(parent_dir + "/python_code")

In [ ]:
from implementation.Cloth import Cloth
from implementation.utils import createRectangularMesh
import numpy as np
np.set_printoptions(threshold=sys.maxsize)
import time

In [ ]:
# Caida libre
na = 2; nb = 3
# na = 4; nb = 7
np.random.seed(1)
X, T = createRectangularMesh(a = 0.5, b = 0.8, na = na, nb = nb, h = 0.2)
X[:,2] += 0.7; 
X += 0.0001*np.random.randn(X.shape[0],3) 

In [ ]:
cloth = Cloth(X, T); 
dt = cloth.estimateTimeStep(L=0.8)

In [ ]:
# solver parameters
dt = 0.002 #time step
tol = 0.0095 # up to 0.75% of relative error in constraint satisfaction to stop iterations

#physical parameters
rho = 0.1 #cloth density
delta = 0.08 # aerodynamics parameter: between 0 and rho
kappa = 0.2*1e-4 # stifness or bending resistance
alpha = 0.2 #damping of oscillations
shr = 1.2*1e-4 #allowed shearing resistance
str = 0.005*1e-4 #allowed stretching resistance
mu_f = 0.45 #friction with the floor
mu_s = 0.35 #friction with the cloth itself
thck = 0.95 #size of the balls

cloth.setSimulatorParameters(dt=dt,tol=tol,
                                rho=rho,delta=delta,kappa=kappa,shr=shr,
                                str=str,alpha=alpha,mu_f=mu_f,mu_s=mu_s,
                                thck=thck)

In [ ]:
tf = int(6/dt)
tf = 1000; t = np.linspace(0,2*np.pi,tf); freq = 3
inds_ctr = [0, na-1]
u = X[inds_ctr]

In [ ]:
print(cloth.A2)

In [ ]:
Xc = cloth.positions
Xc

In [ ]:
F = cloth.faces    
print(f'faces: {F}')
Xf = Xc[F]
Xf

In [ ]:
Xw_face_centers = 0.25 * (cloth.A2 @ cloth.positions)
Xw_face_centers


In [ ]:
box = 0.001 * np.array([3, 30, 6], dtype=float)
half = 0.5 * box
Xc_nodes = cloth.positions

inside_nodes = (
    (np.abs(Xc_nodes[:, 0]) <= half[0]) &
    (np.abs(Xc_nodes[:, 1]) <= half[1]) &
    (np.abs(Xc_nodes[:, 2]) <= half[2])
)


In [ ]:
inside_nodes

In [ ]:
inside = [False, False, False, False, False, True] 
np.where(inside)


In [ ]:
cloth.faces[[0, 1]].reshape(-1)

In [ ]:
A = np.roll(Xf[0], -1, axis=0) # (4, 3)
B = np.repeat(Xw_face_centers[0].reshape(1,3), 4, axis=0) # (4, 3)

triangles = np.stack([Xf[0], A, B], axis=1) # (3, 3, 4)
triangles

In [ ]:
for i in range(1000):
    cloth.simulate(u = u, control = inds_ctr)
u = cloth.positions[inds_ctr]; 
for i in range(tf):
    u[:,1] += 0.0035*np.sin(freq*t[i])
    cloth.simulate(u = u, control = inds_ctr)
inds_ctr = [0]
u = cloth.positions[inds_ctr]; 
for i in range(1000):
    cloth.simulate(u = u, control = inds_ctr)

In [ ]:
print('Average iterations',cloth.total_iters/(len(cloth.history_pos)-1))

In [ ]:
cloth.makeMovie(speed = 6, repeat = True, smooth = 2)

In [ ]:
inds_ctr